# **Installing Required Packages**

In [ ]:
!pip install -q\
sentence-transformers\
chromadb \
whisper\
ffmpeg-python\
pydub\
tqdm\
nltk\
scikit-learn\
sqlite-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━

# **Mounting Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q chromadb sentence-transformers scikit-learn tqdm librosa soundfile

In [ ]:
!pip install -q openai-whisper

# import whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 31.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# **Importing Libraries**

In [ ]:
# Basic library
import sqlite3
import re
import os
import nltk
import chromadb
import whisper
import numpy as np
import pandas as pd
import torch
import math
import random
import zipfile
import io
import torch

from tqdm import tqdm
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from chromadb.config import Settings

# **Loading Data**

In [ ]:
db = '/content/drive/MyDrive/eng_subtitles_database.db'

conn = sqlite3.connect(db)
cursor = conn.cursor()
cursor.execute("SELECT num,name,content FROM zipfiles")

rows=cursor.fetchall()
print('Total Subtitles rows:',len(rows))

Total Subtitles rows: 82498


In [ ]:
rows = cursor.execute("""
    SELECT name, num, content
    FROM zipfiles
    ORDER BY name, num
""").fetchall()

print("Loaded rows:", len(rows))

Loaded rows: 82498


In [ ]:
USE_SAMPLE = True
SAMPLE_PERCENT = 0.30

if USE_SAMPLE:
    sample_size = int(len(rows) * SAMPLE_PERCENT)
    rows = random.sample(rows, sample_size)

print("Using rows:", len(rows))

Using rows: 24749


# **Data Cleaning**

In [ ]:
documents_by_movie = {}

def clean_text(text):
    text = re.sub(r"\d+\n", "", text)
    text = re.sub(r"\d{2}:\d{2}:\d{2}.*\n", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower().strip()

for name, num, zip_bytes in rows:
    if not zip_bytes:
        continue

    try:
        with zipfile.ZipFile(io.BytesIO(zip_bytes), 'r') as z:
            for file_name in z.namelist():

                if file_name.lower().endswith(".srt"):
                    with z.open(file_name) as f:
                        content = f.read().decode("utf-8", errors="ignore")
                        cleaned = clean_text(content)

                        # 🚀 NO LENGTH FILTER NOW
                        if name not in documents_by_movie:
                            documents_by_movie[name] = ""

                        documents_by_movie[name] += " " + cleaned

    except Exception as e:
        print("Error:", name, e)

print("Movies extracted:", len(documents_by_movie))

Movies extracted: 19628


In [ ]:
merged_docs = []

for movie_name, full_text in documents_by_movie.items():
    merged_docs.append((movie_name, full_text))

print("Merged movie documents:", len(merged_docs))

Merged movie documents: 19628


# **Chunking**

In [ ]:
def chunk_text(text, chunk_size=300, overlap=75):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks


documents = []
metadatas = []
ids = []

chunk_counter = 0  # global counter

for movie_name, full_text in merged_docs:
    chunks = chunk_text(full_text)

    for chunk in chunks:
        documents.append(chunk)
        metadatas.append({"movie": movie_name})
        ids.append(chunk_counter)  # unique numeric ID
        chunk_counter += 1

print("Total chunks:", len(documents))
print("Total metadata:", len(metadatas))
print("Total IDs:", len(ids))

Total chunks: 528695
Total metadata: 528695
Total IDs: 528695


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
tfidf = TfidfVectorizer(max_features=50000)
tfidf_matrix = tfidf.fit_transform(documents)

In [ ]:
query = "family secret hidden truth"

q_vec = tfidf.transform([query])
scores = cosine_similarity(q_vec, tfidf_matrix)[0]

top_idx = np.argsort(scores)[-5:][::-1]

print("\nKeyword Search Results:")
for i in top_idx:
    print("Score:", round(scores[i],4))
    print("Movie:", metadatas[i]["movie"])
    print("Text:", documents[i][:250])
    print("-"*50)


Keyword Search Results:
Score: 0.3952
Movie: fosters.home.for.imaginary.friends.s01.e03.the.trouble.with.scribbles.(2004).eng.1cd
Text: the other eduardo so coco have any secret door secrets you d like to share so they want to be all secretive about the secretive secret do they well you can t keep a secret secret from blooregard q kazoo i ve got ways to make em talk you know i was ju
--------------------------------------------------
Score: 0.3042
Movie: cynga.(1992).eng.1cd
Text: name i have to forget why for my own good well for some time anyway careful the border stop or i ll shoot what s that well got him the street remembers mr beck he was so agile your name jozef glazewski the son of grzegorz and genowefa born on novembe
--------------------------------------------------
Score: 0.295
Movie: fosters.home.for.imaginary.friends.s01.e03.the.trouble.with.scribbles.(2004).eng.1cd
Text: mysterious secrets there are secrets behind that door s s secrets preposterous but you just said i s

In [ ]:
print(len(documents))
print(documents[0][:500])

528695
police arrived mid afternoon on a call gathering evidence i remember clearly on this particular day i was in my office and this call came out and we hear that there are three bodies on site so we just responded out to the house when i arrived on scene the media has showed up they re already snapping pictures and what we found in that house is shocking police say vladimir pokhilko used a hammer and hunting knife to kill his wife and son as they slept then used the knife to kill himself it looks l


# **Embedding**

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("intfloat/e5-base-v2", device="cuda")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# **Vector Database**

In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/chroma_subtitle_db"
)

collection = client.get_or_create_collection(
    name="subtitle_semantic",
    metadata={"hnsw:space": "cosine"}
)

print("Already stored:", collection.count())

Already stored: 347328


In [ ]:
BATCH_SIZE = 64
PHASE_SIZE = 800_000

start = collection.count()
end = min(start + PHASE_SIZE, len(documents))

print("Processing:", start, "to", end)

for i in tqdm(range(start, end, BATCH_SIZE)):

    batch_docs = documents[i:i+BATCH_SIZE]
    batch_meta = metadatas[i:i+BATCH_SIZE]
    batch_ids  = ids[i:i+BATCH_SIZE]

    embeddings = embedder.encode(
        batch_docs,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    collection.add(
        documents=batch_docs,
        embeddings=embeddings.tolist(),
        metadatas=batch_meta,
        ids=[str(x) for x in batch_ids]
    )

print("Now stored:", collection.count())

Processing: 0 to 527489


 66%|██████▌   | 5404/8243 [4:49:56<2:29:27,  3.16s/it]

# **Checking for stored vectors**

In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/chroma_subtitle_db"
)

collection = client.get_or_create_collection(
    name="subtitle_semantic",
    metadata={"hnsw:space": "cosine"}
)

print("Already stored:", collection.count())

Already stored: 347328


# **Semantic Search Using Vector Embeddings**

In [ ]:
query = "he is hiding something from the family"

q_emb = embedder.encode(query, normalize_embeddings=True)

results = collection.query(
    query_embeddings=[q_emb],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

In [ ]:
print("\nSemantic Search Results:")

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("\nMovie:", meta["movie"])
    print("Distance:", round(dist,4))
    print("Similarity:", round(1 - dist,4))
    print("Text:", doc[:300])


Semantic Search Results:

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.225
Similarity: 0.775
Text: want me to leave this place but you re worried speak to me answer me why why corazon why what happened lorna what s wrong what happened my blood pressure is getting high you bitch come here come here stay here with aunt aunt you don t have the right to hurt my child you don t have the right to hurt 

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.2291
Similarity: 0.7709
Text: mother that you showed me all of these why she ll spank you why because that s not yours and you showed it to me you know what boyito if your mom and lando find out they will get mad at you it s true you re kind lando it s almost sunset never mind why my money s lacking i wasn t able to borrow from 

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.2304
Similarity: 0.7696
Text: also for being the daughter you talk too much she has just been buried and yet you ve showing the whole world that mother s death h

In [ ]:
query1 = "Mother sick with fever "

q_emb = embedder.encode(query1, normalize_embeddings=True)

results = collection.query(
    query_embeddings=[q_emb],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

In [ ]:
print("\nSemantic Search Results:")

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("\nMovie:", meta["movie"])
    print("Distance:", round(dist,4))
    print("Similarity:", round(1 - dist,4))
    print("Text:", doc[:300])


Semantic Search Results:

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.2071
Similarity: 0.7929
Text: in the parlor you d only go back to hell that you came from to dory to ning ning answer me i m not throwing you out you re doing this to me maybe because you don t love me anymore why don t you tell the truth lorna remember this i have one bad character different form the ones you know if a person d

Movie: ambulance.(2022).eng.1cd
Distance: 0.2142
Similarity: 0.7858
Text: incident copy on scene let s go let me see stay clear stay clear i need to check come on out please no save my baby careful with the girl move move move let me stay with her medic s here let them in is it still there i don t want to look at it hey sweetie sweetie hey just got to look at me you don t

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.2159
Similarity: 0.7841
Text: me that i mustn t come near you to ask for help but i don t have anyone else to run to i feel bad you re not our relative but you care 

# **Speech-to-Text Semantic Retrieval using Whisper and Embeddings**

In [ ]:
model = whisper.load_model("base")

result = model.transcribe("/content/drive/MyDrive/My Audio/WhatsApp Audio 2026-04-14 at 3.55.50 PM.mp4")
query_text = result["text"]

print("Transcribed:", query_text)

Transcribed:  A man treating on his wife.


In [ ]:
query_embedding = embedder.encode(query_text)

In [ ]:
query = "Represent this sentence for searching relevant passages: " + query_text
query_embedding = embedder.encode(query, normalize_embeddings=True)

In [ ]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

In [ ]:
print("\nSemantic Search Results:")

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("\nMovie:", meta["movie"])
    print("Distance:", round(dist,4))
    print("Similarity:", round(1 - dist,4))
    print("Text:", doc[:300])


Semantic Search Results:

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.1941
Similarity: 0.8059
Text: are you doing it s like you re refusing rice what if i burn you come here son have some bottle brought would this be okay hey it s thick hey lito yes buy alcohol and grilled pork hurry up don t forget to give back the change you might keep it again i won t i ll go now come candida candida if you won

Movie: ambulance.(2022).eng.1cd
Distance: 0.2047
Similarity: 0.7953
Text: are you doing randazzo my wife s breaking my balls what are you on the phone for let s go i can t do this shit come on sit down all right sit down you re making me anxious enjoy my break room here it s good coffee huh yeah great look man you can definitely uh tell me to go eff off but uh i could use

Movie: ambulance.(2022).eng.1cd
Distance: 0.2054
Similarity: 0.7946
Text: working drop down to miles an hour give him some distance she s operating on our brother right now cam his blood pressure s dropping he s abo

In [ ]:
model = whisper.load_model("base")

result = model.transcribe("/content/drive/MyDrive/My Audio/WhatsApp Audio 2026-04-14 at 3.56.00 PM.mp4")
query_text2 = result["text"]

print("Transcribed:", query_text2)

Transcribed:  Parents giving surprise to their child.


In [ ]:
query_embedding = embedder.encode(query_text2)

In [ ]:
query = "Represent this sentence for searching relevant passages: " + query_text2
query_embedding = embedder.encode(query, normalize_embeddings=True)

In [ ]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

In [ ]:
print("\nSemantic Search Results:")

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("\nMovie:", meta["movie"])
    print("Distance:", round(dist,4))
    print("Similarity:", round(1 - dist,4))
    print("Text:", doc[:300])


Semantic Search Results:

Movie: roswell.s01.e20.max.to.the.max.(2000).eng.1cd
Distance: 0.1966
Similarity: 0.8034
Text: um sorry he always used to tell me son you got nothin to prove but he was wrong a father sets a fine example like that for his son it s only right he should try to live up to it i suppose that your boy yeah he got any interest in the badge no that s a shame you know you did such a great job on that 

Movie: ambulance.(2022).eng.1cd
Distance: 0.1975
Similarity: 0.8025
Text: just took it yo check this out look what i found look at this dad s booties look at these pictures you and me mmm look at you and who s that wait what class is that um ah ms burns ms burns yeah right after you stood up embarrassed me and told the whole class that your family took me in i owe your fa

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.198
Similarity: 0.802
Text: s here mother this is lorna please open the door oh lorna choleng where s mother she s in there keep knocking she s just s

In [ ]:
model3= whisper.load_model("base")

result = model.transcribe("/content/drive/MyDrive/My Audio/WhatsApp Audio 2026-04-14 at 3.56.08 PM.mp4")
query_text3 = result["text"]

print("Transcribed:", query_text3)

Transcribed:  Drama Between Mother In Law & Daughter In Law


In [ ]:
query_embedding = embedder.encode(query_text3)

In [ ]:
query = "Represent this sentence for searching relevant passages: " + query_text3
query_embedding = embedder.encode(query, normalize_embeddings=True)

In [ ]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

In [ ]:
print("\nSemantic Search Results:")

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("\nMovie:", meta["movie"])
    print("Distance:", round(dist,4))
    print("Similarity:", round(1 - dist,4))
    print("Text:", doc[:300])


Semantic Search Results:

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.1827
Similarity: 0.8173
Text: also for being the daughter you talk too much she has just been buried and yet you ve showing the whole world that mother s death has no value to you i m mourning for her don t you see you re really emotional hey i m already a widow lucy and i will live here to lessen our expenses for house rentals 

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.1846
Similarity: 0.8154
Text: s here mother this is lorna please open the door oh lorna choleng where s mother she s in there keep knocking she s just sulking oh so he s the one you married how did you know through loraine once when she had dory perm her hair he told her that you already got married that you re no longer under t

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.1898
Similarity: 0.8102
Text: advertise your product or brand here contact www opensubtitles org today you came home when you re hungry you always come home he

In [ ]:
model4 = whisper.load_model("base")

result = model.transcribe("/content/drive/MyDrive/My Audio/WhatsApp Audio 2026-04-14 at 3.56.17 PM.mp4")
query_text4 = result["text"]

print("Transcribed:", query_text4)

Transcribed:  Friends hanging out together and enjoying the moment.


In [ ]:
query_embedding = embedder.encode(query_text4)

In [ ]:
query = "Represent this sentence for searching relevant passages: " + query_text4
query_embedding = embedder.encode(query, normalize_embeddings=True)

In [ ]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

In [ ]:
print("\nSemantic Search Results:")

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("\nMovie:", meta["movie"])
    print("Distance:", round(dist,4))
    print("Similarity:", round(1 - dist,4))
    print("Text:", doc[:300])


Semantic Search Results:

Movie: two.and.a.half.men.s10.e05.thats.not.what.they.call.it.in.amsterdam.(2012).eng.1cd
Distance: 0.1964
Similarity: 0.8036
Text: stuff has anything to do with us that s true so what do you say dinner okay to dinner it will be nice to go out with someone normal for a change hello alan rose you ve got a lot of nerve coming to this house as much nerve as you have still living in it touch oh this place looks fantastic don t act l

Movie: two.and.a.half.men.s10.e05.thats.not.what.they.call.it.in.amsterdam.(2012).eng.1cd
Distance: 0.2003
Similarity: 0.7997
Text: m just gonna be alone for a while and if the real thing comes along i ll know it that s true i can still remember the exact instant that i knew lyndsey was the one how d you know she said i give up you re the one hey walden hey you here by yourself yes i m alone and i plan to stay that way oh well i

Movie: ambulance.(2022).eng.1cd
Distance: 0.2134
Similarity: 0.7866
Text: coming over let me see if i can

In [ ]:
model5 = whisper.load_model("base")

result = model.transcribe("/content/drive/MyDrive/My Audio/WhatsApp Audio 2026-04-14 at 3.56.25 PM.mp4")
query_text5 = result["text"]

print("Transcribed:", query_text5)

Transcribed:  Child is crying and mother searching for her child.


In [ ]:
query_embedding = embedder.encode(query_text5)

In [ ]:
query = "Represent this sentence for searching relevant passages: " + query_text5
query_embedding = embedder.encode(query, normalize_embeddings=True)

In [ ]:
query_embedding = embedder.encode(query_text)

In [ ]:
query = "Represent this sentence for searching relevant passages: " + query_text
query_embedding = embedder.encode(query, normalize_embeddings=True)

In [ ]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

In [ ]:
print("\nSemantic Search Results:")

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("\nMovie:", meta["movie"])
    print("Distance:", round(dist,4))
    print("Similarity:", round(1 - dist,4))
    print("Text:", doc[:300])


Semantic Search Results:

Movie: tinik.sa.dibdib.(1985).eng.1cd
Distance: 0.1941
Similarity: 0.8059
Text: are you doing it s like you re refusing rice what if i burn you come here son have some bottle brought would this be okay hey it s thick hey lito yes buy alcohol and grilled pork hurry up don t forget to give back the change you might keep it again i won t i ll go now come candida candida if you won

Movie: ambulance.(2022).eng.1cd
Distance: 0.2047
Similarity: 0.7953
Text: are you doing randazzo my wife s breaking my balls what are you on the phone for let s go i can t do this shit come on sit down all right sit down you re making me anxious enjoy my break room here it s good coffee huh yeah great look man you can definitely uh tell me to go eff off but uh i could use

Movie: ambulance.(2022).eng.1cd
Distance: 0.2054
Similarity: 0.7946
Text: working drop down to miles an hour give him some distance she s operating on our brother right now cam his blood pressure s dropping he s abo

# **The End**
